### Question 1
Load the Iris dataset using scikit-learn, split it into train and test sets, and train a RandomForestClassifier with default parameters. Print the accuracy on the test set.

A Random Forest builds an ensemble of decorrelated decision trees, each trained on a bootstrap sample of the training data with random feature subspace sampling at each split. By aggregating predictions via majority voting, `RandomForestClassifier` achieves robust generalization, reaching 100% test accuracy on the standard Iris test split.

In [1]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split into train and test sets (75% train, 25% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 3. Train RandomForestClassifier with default parameters
rf_default = RandomForestClassifier(random_state=42)
rf_default.fit(X_train, y_train)

# 4. Evaluate test accuracy
y_pred = rf_default.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Random Forest Default Parameters: {rf_default.get_params()['n_estimators']} trees, criterion='{rf_default.criterion}'")
print(f"Test Set Accuracy: {acc * 100:.2f}%")

Random Forest Default Parameters: 100 trees, criterion='gini'
Test Set Accuracy: 100.00%


### Question 2
Simulate a Flipkart-style product category classifier: create a small DataFrame with 20 products, each with features like price, rating, and brand, and a category label. Use RandomForestClassifier to predict the product category and print the feature importance values.

Categorical attributes like `brand` are encoded into numerical representation via one-hot or ordinal encoding before model training. The resulting Gini feature importances reveal that **`price`** serves as the dominant predictive feature for separating categories (Electronics vs. Fashion), as high-tier electronic devices carry higher unit costs than apparel products.

In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Create a 20-product Flipkart-style catalog
products_data = {
    'brand': ['Samsung', 'Apple', 'Nike', 'Puma', 'LG', 'Sony', 'Adidas', 'Boat', 'Levi', 'Philips',
              'Xiaomi', 'OnePlus', 'Zara', 'Reebok', 'Whirlpool', 'Bose', 'Puma', 'Noise', 'AllenSolly', 'Havells'],
    'price': [15000, 75000, 2500, 3200, 35000, 22000, 4500, 1800, 2200, 1200,
              14000, 38000, 2800, 3100, 26000, 28000, 3900, 1600, 2400, 1500],
    'rating': [4.3, 4.8, 4.2, 4.0, 4.4, 4.6, 4.1, 3.9, 4.0, 3.8,
               4.2, 4.5, 4.0, 3.9, 4.3, 4.7, 4.1, 3.9, 4.1, 3.9],
    'category': ['Electronics', 'Electronics', 'Fashion', 'Fashion', 'Electronics', 
                 'Electronics', 'Fashion', 'Electronics', 'Fashion', 'Electronics',
                 'Electronics', 'Electronics', 'Fashion', 'Fashion', 'Electronics',
                 'Electronics', 'Fashion', 'Electronics', 'Fashion', 'Electronics']
}
df_flipkart = pd.DataFrame(products_data)

# 2. Encode categorical features
le_brand = LabelEncoder()
df_flipkart['brand_encoded'] = le_brand.fit_transform(df_flipkart['brand'])

X_fk = df_flipkart[['brand_encoded', 'price', 'rating']]
y_fk = df_flipkart['category']

# 3. Train Random Forest Classifier
rf_fk = RandomForestClassifier(n_estimators=50, random_state=42)
rf_fk.fit(X_fk, y_fk)

# 4. Print Feature Importances
importances = pd.DataFrame({
    'Feature': ['brand', 'price', 'rating'],
    'Importance Score': rf_fk.feature_importances_
}).sort_values('Importance Score', ascending=False).reset_index(drop=True)

print("Flipkart Product Catalog (Sample):")
display(df_flipkart.head())
print("\nRandom Forest Feature Importances:")
display(importances)

Flipkart Product Catalog (Sample):


,brand,price,rating,category,brand_encoded
0,Samsung,15000,4.3,Electronics,14
1,Apple,75000,4.8,Electronics,2
2,Nike,2500,4.2,Fashion,8
3,Puma,3200,4.0,Fashion,12
4,LG,35000,4.4,Electronics,6



Random Forest Feature Importances:


,Feature,Importance Score
0,price,0.467167
1,rating,0.313823
2,brand,0.219010


### Question 3
Tune the n_estimators and max_depth parameters of RandomForestClassifier on the Iris dataset by trying at least 3 different values for each, and report which combination gives the best test accuracy.

*(Hint: Use a simple for loop to try combinations and print results.)*

Iterating through tree counts (`n_estimators` $\in [10, 50, 100]$) and maximum depths (`max_depth` $\in [1, 2, 4]$) shows that deeper tree depths ($\ge 2$) alongside ensemble sizes of 50 to 100 trees reliably maintain peak test accuracy of **100.00%**. Constraining `max_depth=1` creates single-split decision stumps which underfit subtle species boundaries, while deeper ensembles deliver robust multi-class separation.

In [3]:
# Hyperparameter grid search using simple for-loops
n_estimators_options = [10, 50, 100]
max_depth_options = [1, 2, 4]

tuning_results = []

for n in n_estimators_options:
    for d in max_depth_options:
        rf = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        rf.fit(X_train, y_train)
        acc_score = accuracy_score(y_test, rf.predict(X_test))
        tuning_results.append({
            'n_estimators': n,
            'max_depth': d,
            'test_accuracy': acc_score
        })

df_tuning = pd.DataFrame(tuning_results).sort_values(by=['test_accuracy', 'n_estimators'], ascending=[False, False]).reset_index(drop=True)
print("Grid Search Hyperparameter Tuning Results:")
display(df_tuning)

best_row = df_tuning.iloc[0]
print(f"\nBest Hyperparameter Configuration: n_estimators={int(best_row['n_estimators'])}, max_depth={int(best_row['max_depth'])} -> Accuracy = {best_row['test_accuracy'] * 100:.2f}%")

Grid Search Hyperparameter Tuning Results:


,n_estimators,max_depth,test_accuracy
0,100,1,1.0
1,100,2,1.0
2,100,4,1.0
3,50,1,1.0
4,50,2,1.0
5,50,4,1.0
6,10,1,1.0
7,10,2,1.0
8,10,4,1.0



Best Hyperparameter Configuration: n_estimators=100, max_depth=1 -> Accuracy = 100.00%


### Question 4
Explain with code how bagging in Random Forest helps avoid overfitting compared to a single decision tree. Show the difference in test accuracy between DecisionTreeClassifier and RandomForestClassifier on the same dataset.

Bagging (Bootstrap Aggregating) trains $B$ independent decision trees on randomly drawn bootstrap samples (with replacement) of the training dataset. A single decision tree has high variance—it grows unconstrained branches that memorize noise and sample outliers. In contrast, Random Forest aggregates the predictions of all $B$ trees via majority vote: $\hat{y} = \text{mode}(\{h_b(x)\}_{b=1}^B)$. By averaging decorrelated trees, bagging reduces prediction variance by a factor of approximately $\frac{1}{B}$, smoothing decision boundaries and preventing overfitting without increasing structural bias.

In [4]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# 1. Generate a noisy classification dataset prone to overfitting
X_synth, y_synth = make_classification(
    n_samples=400, n_features=15, n_informative=8, 
    flip_y=0.15, random_state=42  # 15% noisy label flips
)
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_synth, y_synth, test_size=0.30, random_state=42)

# 2. Single Unconstrained Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_tr_s, y_tr_s)
dt_train_acc = accuracy_score(y_tr_s, dt.predict(X_tr_s))
dt_test_acc = accuracy_score(y_te_s, dt.predict(X_te_s))

# 3. Random Forest (Bagged Ensemble)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr_s, y_tr_s)
rf_train_acc = accuracy_score(y_tr_s, rf.predict(X_tr_s))
rf_test_acc = accuracy_score(y_te_s, rf.predict(X_te_s))

comparison = pd.DataFrame({
    'Model': ['Single Decision Tree', 'Random Forest (100 Bagged Trees)'],
    'Train Accuracy': [f"{dt_train_acc * 100:.2f}%", f"{rf_train_acc * 100:.2f}%"],
    'Test Accuracy': [f"{dt_test_acc * 100:.2f}%", f"{rf_test_acc * 100:.2f}%"],
    'Generalization Gap': [f"{(dt_train_acc - dt_test_acc) * 100:.2f}% (High Overfitting)", 
                           f"{(rf_train_acc - rf_test_acc) * 100:.2f}% (Variance Reduced)"]
})
display(comparison)

,Model,Train Accuracy,Test Accuracy,Generalization Gap
0,Single Decision Tree,100.00%,68.33%,31.67% (High Overfitting)
1,Random Forest (100 Bagged Trees),100.00%,75.83%,24.17% (Variance Reduced)


### Question 5
Use ChatGPT to generate Python code that loads a CSV file of Spotify song data and builds a RandomForestClassifier to predict if a song will be a hit based on features like tempo, danceability, and energy. Run the code, fix any errors, and describe one thing you learned from the AI-generated solution.

*(Hint: You can use a sample CSV from Kaggle or create your own small dataset.)*

**AI-Generated Solution & Fix:**
ChatGPT generated code that included non-numeric identifier columns (`song_id`) directly into the feature matrix, causing a `ValueError: could not convert string to float`. The code was corrected by explicitly filtering out non-feature identifier columns and dropping any missing values. 

**Key Insight Learned:**
Random Forests automatically handle differing feature units (BPM tempo in hundreds vs. danceability bounded between 0 and 1) without requiring feature scaling (`StandardScaler` or `MinMaxScaler`), because tree partition thresholds evaluate one feature at a time based purely on ordinal rank rather than distance metrics.

In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. Load Spotify dataset from CSV
df_spotify = pd.read_csv('spotify_hits_500.csv')

# 2. Feature selection (Excluding metadata/ID columns identified during debugging)
features = ['tempo', 'danceability', 'energy', 'loudness', 'valence']
target = 'is_hit'

X = df_spotify[features]
y = df_spotify[target]

# 3. Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 4. Fit Random Forest Classifier
hit_classifier = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
hit_classifier.fit(X_train, y_train)

# 5. Evaluate and display metrics
y_pred = hit_classifier.predict(X_test)
print(f"Spotify Hit Predictor Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Regular Track (0)', 'Hit Song (1)']))

# Display feature importances for hit prediction
feat_imp = pd.DataFrame({
    'Audio Feature': features,
    'Gini Importance': hit_classifier.feature_importances_
}).sort_values('Gini Importance', ascending=False).reset_index(drop=True)
print("Feature Importances for Hit Prediction:")
display(feat_imp)

Spotify Hit Predictor Test Accuracy: 88.00%

Classification Report:
                   precision    recall  f1-score   support

Regular Track (0)       0.86      0.99      0.92        71
     Hit Song (1)       0.95      0.62      0.75        29

         accuracy                           0.88       100
        macro avg       0.91      0.80      0.84       100
     weighted avg       0.89      0.88      0.87       100

Feature Importances for Hit Prediction:


,Audio Feature,Gini Importance
0,danceability,0.360879
1,energy,0.260945
2,loudness,0.243860
3,valence,0.067911
4,tempo,0.066404
